# Generalized spectral clustering

Clusters all 445 feature titles together, without filtering by product type.

It starts from the shared similarity matrix: `3_similarity.ipynb` now saves it to `data/similarity.npz`, and this notebook (like the agglomerative and DBSCAN ones) just loads it. Every method uses the same input, so differences in the results come from the algorithm.

**Method** (normalized spectral clustering):
1. We build a graph from the similarity matrix: attributes are nodes, similarities are edge weights (each node keeps its `K_NEIGHBORS` most similar neighbours).
2. We compute the normalized Laplacian and its eigenvectors with the smallest eigenvalues.
3. Finally, we run k-means on those eigenvectors

In [ ]:
import json
import time
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score

K_NEIGHBORS = 15 
N_EIGEN = 60
K_RANGE = range(5, 61, 5)
SEED = 0


with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"
print(f"Loaded {n} attributes, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


## Build the similarity graph

The diagonal is set to 0 (no self-loops). With `K_NEIGHBORS`, each attribute keeps only its most similar neighbours; the result is symmetrised so that an edge is kept if either endpoint selects it. This makes the graph sparse, removes weak noisy links, and is what makes spectral clustering scale to larger catalogs.

In [ ]:
def build_affinity(S, k_neighbors=None):
    W = similarity.copy()
    np.fill_diagonal(W, 0)
    threshold = np.sort(W, axis=1)[:, -K_NEIGHBORS][:, None]       # each row's 15th-highest similarity
    W = np.where((W >= threshold) | (W >= threshold.T), W, 0) 
    return W


W = build_affinity(similarity, K_NEIGHBORS)
print(f"Edges kept: {np.count_nonzero(np.triu(W))}, isolated nodes: {int((W.sum(axis=1) == 0).sum())}")

Edges kept: 4617, isolated nodes: 0


## Normalized Laplacian and eigengap

`np.linalg.eigh` is used because $L_{sym}$ is symmetric; it returns eigenvalues in ascending order. Small eigenvalues correspond to well-separated groups; the largest jump (eigengap) among them suggests the number of clusters.

In [37]:
def spectral_embedding(W, n_eigen):
    degree = W.sum(axis=1)
    d_inv_sqrt = 1 / np.sqrt(np.maximum(degree, 1e-12))
    L_sym = np.eye(len(W)) - d_inv_sqrt[:, None] * W * d_inv_sqrt[None, :]
    eigenvalues, eigenvectors = np.linalg.eigh(L_sym)
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]


eigenvalues, eigenvectors = spectral_embedding(W, N_EIGEN)

K_MIN, K_MAX = 20, 80

gaps = np.diff(eigenvalues)
k_eigengap = int(np.argmax(gaps[K_MIN - 1:K_MAX]) + K_MIN)
print(f"Eigengap between k={K_MIN} and {K_MAX} suggests k = {k_eigengap}")

Eigengap between k=20 and 80 suggests k = 25


## 4. Spectral clustering

Rows of the first *k* eigenvectors are normalized to unit length (Ng–Jordan–Weiss step), then clustered with k-means.

In [38]:
def spectral_cluster(eigenvectors, k, seed=SEED):
    U = eigenvectors[:, :k]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(U)
    return labels, U


k = k_eigengap
labels, U = spectral_cluster(eigenvectors, k)
sizes = Counter(labels)
print(f"{k} clusters; sizes (largest first): {[size for _, size in sizes.most_common()]}")

25 clusters; sizes (largest first): [30, 30, 26, 24, 24, 21, 20, 20, 19, 19, 19, 19, 18, 17, 17, 17, 14, 14, 14, 13, 12, 11, 11, 9, 7]


## 5. Evaluation

TO BE COMPLETED

## 6. Cluster visualization

t-SNE projects the same precomputed distances (`1 - similarity`) to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity, but the projection can distort some distances. The largest groups get distinct colours, the others are grey, and singletons (if any) are hollow markers. Hover over a point to see its attribute and group.

In [39]:
from sklearn.manifold import TSNE

HIGHLIGHT_GROUPS = 8
HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

distance_matrix = 1 - similarity
multi = [label for label, size in sizes.items() if size > 1]

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar attribute as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1], dtype=int)

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Attribute clusters: spectral clustering (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))